## 4.2 RNN网络结构 - 全连接层

#### 1. 为什么 RNN 层之后还需要全连接层？

##### 1.1 RNN层负责的是“提取时序特征”

在上一节中我们已经知道，RNN层最核心的工作是：

> 根据当前输入 $x_t$ 和上一时刻隐藏状态 $h_{t-1}$，计算当前隐藏状态 $h_t$

也就是：

$h_t = f(W_x x_t + W_h h_{t-1} + b)$

这里得到的 $h_t$，本质上是：

> 模型在当前时间步对历史信息的一种内部表示。

它更像是一种“特征”或“记忆”，而不是最终任务答案。

也就是说：

* RNN层负责提取时序信息
* 但它还没有把这种信息映射成最终想要的输出结果

##### 1.2 最终任务往往需要一个“结果空间”

比如不同任务中，我们真正想要的可能是：

* 一个类别
* 一个概率分布
* 一个连续数值
* 一个词表中的某个词

这些结果通常都不直接等于隐藏状态 $h_t$

因为隐藏状态的维度是模型自己设定的，比如：

* 16维
* 32维
* 128维

但我们的任务输出空间往往是由任务决定的，比如：

* 二分类：输出 $1$ 个值或 $2$ 个类别分数
* 多分类：输出 $C$ 个类别分数
* 词预测：输出整个词表大小对应的分数

所以我们需要再加一层，把隐藏状态映射到任务需要的输出空间。  
这层通常就是：

> 全连接层（Fully Connected Layer / Linear Layer）

##### 1.3 所以完整结构应该分成两部分

一个基础的 RNN 模型，结构上通常可以拆成：

输入序列 $\rightarrow$ RNN层 $\rightarrow$ 隐藏状态 $\rightarrow$ 全连接层 $\rightarrow$ 最终输出

也就是说：

* RNN层负责“理解序列”
* 全连接层负责“生成任务结果”

这一点一定要分清楚。✅

#### 2. 全连接层在这里到底起什么作用？

##### 2.1 全连接层的本质：做线性映射

全连接层最基础的形式我们以前已经学过：

$y = Wx + b$

在全连接层中，这里的输入不再是原始特征，而是：

> RNN层输出的隐藏状态 $h_t$

所以全连接层在这里做的事情就是：

> 把隐藏状态映射成当前任务需要的输出。

##### 2.2 全连接层计算公式

如果在第 $t$ 个时间步，我们把隐藏状态 $h_t$ 输入到全连接层，那么最常见的写法是：

$o_t = W_y h_t + b_y$

这里：

* $h_t$：RNN层给出的隐藏状态
* $W_y$：隐藏状态到输出层的权重矩阵
* $b_y$：输出层偏置
* $o_t$：输出层线性结果

这个 $o_t$ 还不一定是最终可解释结果，它往往只是：

> 全连接层输出的线性分数（logits）

如果任务需要概率，还要继续经过激活函数。

##### 2.3 为什么这里叫“全连接层”？

因为全连接层里：

> 输出层中的每一个神经元，都会和输入向量 $h_t$ 的所有维度相连接。

假设：

* 隐藏状态 $h_t$ 是 $8$ 维
* 输出要映射到 $3$ 维

那么全连接层本质上就是：

* 输入 $8$ 个特征
* 输出 $3$ 个结果
* 每个输出都由全部 $8$ 个输入共同决定

所以这就是标准的全连接映射。


#### 3. RNN层和全连接层之间是什么关系？

##### 3.1 RNN层先负责产生隐藏状态

RNN层的核心递推公式是：

$h_t = f(W_x x_t + W_h h_{t-1} + b)$

这一步做完后，我们得到的是：

> 当前时间步的隐藏状态 $h_t$

它表示：

> 到当前时刻为止，模型对历史序列信息的内部总结。

##### 3.2 全连接层再负责把隐藏状态变成输出结果

然后我们再把这个隐藏状态送入全连接层：

$o_t = W_y h_t + b_y$

如果还需要非线性激活，就继续：

$y_t = g(o_t)$

所以从结构上讲：

> RNN层负责“生成特征”，全连接层负责“解释特征”。

##### 3.3 这两个层分工不同

你可以这样理解：

* RNN层负责回答：  
  “到当前时刻为止，这段序列的信息该如何表示？”
* 全连接层负责回答：  
  “基于这个表示，当前任务结果应该输出什么？”

所以这两层虽然连在一起，但职责完全不同。


#### 4. 为什么理论里常写成 $y_t$，但实际上要分两层理解？

##### 4.1 理论推导里常把整个过程写成一条链

在很多教材里，经常会把 RNN 单步过程写成：

$x_t \rightarrow h_t \rightarrow y_t$

看起来好像：

> RNN直接算出了 $y_t$

但实际上，这只是为了画图方便、公式简化。

##### 4.2 更准确的结构应该是

真正更清楚的写法应该是：

$x_t,\ h_{t-1} \rightarrow h_t \rightarrow o_t \rightarrow y_t$

其中：

* $h_t$：RNN层产生的隐藏状态
* $o_t = W_y h_t + b_y$：全连接层的线性输出
* $y_t = g(o_t)$：最终输出

也就是说：

> 理论里的 $y_t$，其实已经默认包含了“隐藏状态之后再经过输出层”的这一步。

##### 4.3 这也是为什么在 PyTorch 中会容易混淆

因为在 PyTorch 的 `nn.RNN` 中，模块本身只返回：

* 所有时间步的隐藏状态序列
* 最后时刻隐藏状态

它不会自动帮你做：

$o_t = W_y h_t + b_y$

所以如果你要完成分类或预测任务，通常还要自己再写一个：

`nn.Linear(hidden_size, output_size)`

这就是我们这里说的全连接层。



#### 5. 全连接层的输出到底表示什么？

##### 5.1 先看线性输出 $o_t$

公式：

$o_t = W_y h_t + b_y$

这个 $o_t$ 一般表示：

> 当前时间步在输出空间上的线性打分结果

例如：

* 二分类时，可能输出一个分数
* 多分类时，可能输出每个类别一个分数
* 词预测时，可能输出对整个词表中每个词的分数

这时的 $o_t$ 往往还不是最终概率，只是 logits。

##### 5.2 最终输出 $y_t$ 通常要结合任务再决定

根据任务不同，后面可能接不同的激活函数。

* 二分类  
  可以接 sigmoid：  
  $y_t = \sigma(o_t)$
* 多分类  
  可以接 softmax：  
  $y_t = softmax(o_t)$
* 回归  
  可能不接额外激活，直接输出：  
  $y_t = o_t$

所以：

> 全连接层负责映射，最终激活函数负责把映射结果变成适合任务解释的形式。

#### 6. 本节总结 🧠

这一节最核心的逻辑可以总结为：

* RNN层先根据 $x_t$ 和 $h_{t-1}$ 计算隐藏状态 $h_t$
* 隐藏状态 $h_t$ 本质上是时序信息的内部表示
* 全连接层再把 $h_t$ 映射到任务需要的输出空间
* 如果需要，还会继续经过激活函数，得到最终输出 $y_t$

所以完整流程更准确地写成：

$x_t,\ h_{t-1} \rightarrow h_t \rightarrow o_t \rightarrow y_t$

其中：

* RNN层负责“理解序列”
* 全连接层负责“生成结果”